# Nexa HU Scratch — nulláról betanított magyar modell

Ez a notebook **nem finomhangol egy meglévő modellt**. A súlyokat véletlen értékekről indítja, saját BPE tokenizálót tanít magyar szövegekre, majd két szakaszban tanítja be a modellt:

1. **magyar szöveges előképzés** — magyar Wikipédia,
2. **utasítás-követő tanítás** — saját magyar minták, Aya hu és OPUS-100.

**Mit jelent itt a „nulláról”?**

- saját tokenizáló és saját szókincs (16000 token);
- véletlen inicializálású transzformer, nem Qwen/Llama finomhangolás;
- minden modellsúlyt a mi adatainkból tanítunk be;
- ONNX → 4 bites `MatMulNBits` → saját Hugging Face repo → offline az appban.

**Őszinte figyelmeztetés:** egy 30–100 millió paraméteres, nulláról tanított modell **nem ér fel** a Qwen3-0.6B finomhangolásához, és nem lesz ChatGPT-szintű. Felismerhető magyar szöveget, rövid, befejezett válaszokat és egyszerű utasításokat fog tudni. Ha a cél a *lehető legjobb magyar tudás telefoton*, akkor a `nexa_train.ipynb` (Qwen3 alapú finomhangolás) erősebb. Ha a cél a **teljesen saját súly**, akkor ez a notebook a megoldás.

**Futtatás:** Colab → `Futtatás → Futtatókörnyezet módosítása → T4 GPU`, majd celláról cellára. Becsült idő: **60–90 perc**.

**Előfeltétel:** Hugging Face *write* token, Colab `Secrets` paneljén `HF_TOKEN` néven (vagy a notebook kérni fogja).

In [ ]:
# 1. Beállítások
import os
import json
import random
import numpy as np

OUTPUT_REPO   = "sziriusz-blackalt/nexa-hu"   # ide kerül a kész modell
VOCAB_SIZE    = 16000      # saját magyar BPE szókincs
MAX_LEN       = 512        # egy tanítási blokk hossza
N_POSITIONS   = 1024
N_LAYER       = 8
N_HEAD        = 8
N_EMBD        = 512        # kb. 34M-paraméteres modell
RAW_CHUNKS    = 8000       # ~4 millió token magyar Wikipédiából
PRETRAIN_EPOCHS = 1
INSTRUCT_EPOCHS = 2
BATCH         = 8
GRAD_ACCUM    = 8
LR_PRE        = 3e-4
LR_INS        = 1e-4

assert N_EMBD % N_HEAD == 0, "N_EMBD osztható legyen N_HEAD-tel"

random.seed(42)
np.random.seed(42)

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Hugging Face write token: ")

import huggingface_hub
huggingface_hub.login(token=HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN
print("Cél repo:", OUTPUT_REPO)
print("Modell mérete: kb.", round((VOCAB_SIZE * N_EMBD + N_POSITIONS * N_EMBD + N_LAYER * (4 * N_EMBD * N_EMBD + 4 * N_EMBD * N_EMBD)) / 1e6, 1), "M paraméter")

In [ ]:
# 2. Függőségek
!pip -q install --upgrade transformers datasets tokenizers onnx onnxruntime optimum onnxruntime-tools huggingface_hub accelerate

In [ ]:
# 3. Magyar szövegek és a SAJÁT tokenizáló betanítása
from datasets import load_dataset
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers

texts = []

# Magyar Wikipédia a nyelvtudáshoz
try:
    wiki = load_dataset("wikimedia/wikipedia", "20231101.hu", split="train[:30000]")
    for r in wiki:
        t = (r.get("text") or "").strip()
        if len(t) > 300:
            texts.append(t)
    print("Magyar Wikipédia cikkek:", len(texts))
except Exception as e:
    print("Wikipédia kihagyva:", str(e)[:180])

# Extra magyar mondatok, ha a Wikipédia épp nem érhető el
texts.extend([
    "Szia! Jól vagyok, köszönöm. Neked hogy megy?",
    "Nexa AI vagyok, egy magyar nyelvű segéd, amely a telefonodon, internet nélkül is működik.",
    "A magyar nyelv bonyolult nyelvtannal és gazdag szókincsel rendelkezik.",
    "Budapest Magyarország fővárosa, a Duna két partján fekszik.",
    "A nap ma süt, az ég kék, és a madarak énekelnek a kertben.",
])

PAD_TOK, UNK_TOK, BOS_TOK, EOS_TOK = "<|pad|>", "<|unk|>", "<|bos|>", "<|eos|>"

tok = Tokenizer(models.BPE(unk_token=UNK_TOK))
tok.normalizer = normalizers.NFC()
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=[PAD_TOK, UNK_TOK, BOS_TOK, EOS_TOK],
)
tok.train_from_iterator(texts, trainer=trainer, length=len(texts))

PAD_ID = tok.token_to_id(PAD_TOK)
UNK_ID = tok.token_to_id(UNK_TOK)
BOS_ID = tok.token_to_id(BOS_TOK)
EOS_ID = tok.token_to_id(EOS_TOK)
print("Saját szókincs:", tok.get_vocab_size(), "token")
print("Speciális tokenek:", PAD_ID, UNK_ID, BOS_ID, EOS_ID)

In [ ]:
# 4. Saját utasítások + nyílt magyar párok
import json
from datasets import Dataset

SEED_JSONL = r"""
{"instruction":"Szia, hogy vagy?","response":"Szia! Jól vagyok, köszönöm. Neked hogy megy?"}
{"instruction":"Hogy vagy?","response":"Jól vagyok, köszönöm! Szólj, ha segíthetek valamiben."}
{"instruction":"Jó reggelt!","response":"Jó reggelt! Mivel tudok segíteni ma?"}
{"instruction":"Ki vagy te?","response":"Nexa AI vagyok, egy magyar nyelvű segéd, amely a telefonodon, internet nélkül is működik."}
{"instruction":"Milyen modell vagy?","response":"Egy nulláról, magyar adatokkal betanított kis modell vagyok. A súlyaimat a Nexa AI projekt tanította be."}
{"instruction":"Használsz internetet?","response":"Nem küldök adatot sehova. A modell a telefonodon fut, a letöltés után offline is működik."}
{"instruction":"Gyűjtöd a személyes adataimat?","response":"Nem. Minden csevegés csak a készülékeden marad, és nem kerül szerverre."}
{"instruction":"Tudsz angolul?","response":"Igen, de a magyar válaszokban vagyok a legjobb."}
{"instruction":"Milyen nyelven válaszolsz?","response":"Alapból magyarul, természetesen. Ha kéred, angolra is váltok."}
{"instruction":"Mi a fővárosa Magyarországnak?","response":"Magyarország fővárosa Budapest."}
{"instruction":"Mennyi 12 szorozva 7?","response":"12 szorozva 7 az 84."}
{"instruction":"Oldd meg: 2x + 4 = 10","response":"2x + 4 = 10, tehát 2x = 6, így x = 3."}
{"instruction":"Mi a különbség a méter és a kilométer között?","response":"Egy kilométer ezer méter. A méter az alapegység, a kilométer a nagyobb távolságoké."}
{"instruction":"Foglalj össze egy mondatban: A nap süt, az ég kék, madarak énekelnek.","response":"Szép nappal van: süt a nap, kék az ég, és madarak énekelnek."}
{"instruction":"Fordítsd magyarra: Good morning, how are you?","response":"Jó reggelt, hogy vagy?"}
{"instruction":"Fordítsd magyarra: I would like a coffee, please.","response":"Egy kávét kérnék, köszönöm."}
{"instruction":"Fordítsd angolra: Köszönöm a segítséget.","response":"Thank you for your help."}
{"instruction":"Írj egy rövid szép mondatot az őszről!","response":"Az ősz aranyszínű levelei csendesen hullanak a hidegebb szélben."}
{"instruction":"Hogyan kezdjek el programozni?","response":"Kezdd Pythonnal: tanuld meg a változókat, a feltételeket és a ciklusokat, majd írj kis gyakorlóprogramokat."}
{"instruction":"Mire való a CSS?","response":"A CSS az oldal megjelenését írja le: színeket, betűket, térközöket és elrendezést."}
{"instruction":"Mi a Fibonacci-sorozat?","response":"Olyan számsor, ahol minden tag az előző kettő összege: 0, 1, 1, 2, 3, 5, 8."}
{"instruction":"Sorolj fel három begyűrűzőt Magyarországról!","response":"Szilárd Leó, Wigner Jenő és Teller Ede."}
{"instruction":"Miért fontos az offline működés?","response":"Mert így a modell nem küld adatot szerverre, és gyenge hálózat mellett is használható."}
{"instruction":"Hogyan készítek el egy beszédet?","response":"Válassz egy fő üzenetet, írj három pontot alá, és gyakorold rövidre szabva."}
{"instruction":"Mi a teendő, ha az app üres választ ad?","response":"Próbáld újra rövidebb kéréssel, és ellenőrizd a Beállításokban, hogy be van-e töltve a modell."}
"""

records = [json.loads(line) for line in SEED_JSONL.strip().splitlines() if line.strip()]
print("Saját magyar példák:", len(records))

# Aya Dataset – magyar emberi utasítás-válasz párok
try:
    aya = load_dataset("CohereLabs/aya_dataset", split="train")
    aya_hu = aya.filter(lambda r: r["language_code"] == "hun")
    for r in aya_hu:
        ins = (r.get("inputs") or "").strip()
        res = (r.get("targets") or "").strip()
        if ins and res:
            records.append({"instruction": ins, "response": res})
    print("Aya magyar példák:", len(aya_hu))
except Exception as e:
    print("Aya kihagyva:", str(e)[:180])

# OPUS-100 angol-magyar fordítási párok
try:
    opus = load_dataset("Helsinki-NLP/opus-100", "en-hu", split="train[:10000]")
    added = 0
    for r in opus:
        tr = r["translation"]
        en, hu = tr["en"].strip(), tr["hu"].strip()
        if not en or not hu or len(en) > 220 or len(hu) > 220:
            continue
        records.append({"instruction": "Fordítsd magyarra: " + en, "response": hu})
        records.append({"instruction": "Translate to English: " + hu, "response": en})
        added += 2
    print("OPUS fordítási példák:", added)
except Exception as e:
    print("OPUS kihagyva:", str(e)[:180])

seen, clean = set(), []
for r in records:
    ins, res = r["instruction"].strip(), r["response"].strip()
    if not ins or not res or len(ins) > 1200 or len(res) > 1200 or (ins, res) in seen:
        continue
    seen.add((ins, res))
    clean.append({"instruction": ins, "response": res})
records = clean
print("Összes utasítási példa:", len(records))

In [ ]:
# 5. Adatok becsomagolása: nyers blokkok + utasítások
from datasets import Dataset

SYSTEM = ("Te Nexa AI vagy, egy magyar nyelvű, offline segéd. "
          "Válaszolj hibátlan, természetes magyarul. Légy rövid, de teljes: "
          "fejezd be a gondolatot. Ne találj ki tényeket. Ha nem tudod, mondd, hogy nem tudod.")

def build_raw_ds():
    rows, buf = [], []
    for text in texts:
        buf.extend(tok.encode(text).ids)
        while len(buf) >= MAX_LEN and len(rows) < RAW_CHUNKS:
            chunk = buf[:MAX_LEN]
            rows.append({
                "input_ids": list(chunk),
                "labels": list(chunk),
                "attention_mask": [1] * MAX_LEN,
            })
            buf = buf[MAX_LEN:]
        if len(rows) >= RAW_CHUNKS:
            break
    return Dataset.from_list(rows)

def to_example(r):
    prompt = f"System: {SYSTEM}\nUser: {r['instruction']}\nAssistant:"
    full = prompt + " " + r["response"] + EOS_TOK
    p_ids = tok.encode(prompt).ids
    f_ids = tok.encode(full).ids
    if f_ids[:len(p_ids)] != p_ids:
        p_ids = f_ids[:max(1, len(f_ids) - 1)]
    labels = [-100] * len(p_ids) + f_ids[len(p_ids):]
    ids = f_ids[:MAX_LEN]
    labels = labels[:MAX_LEN]
    return {"input_ids": ids, "labels": labels, "attention_mask": [1] * len(ids)}

raw_ds = build_raw_ds()
inst_ds = Dataset.from_list([to_example(r) for r in records])
inst_ds = inst_ds.filter(lambda r: len(r["input_ids"]) > 8)
print("Nyers szövegblokkok:", len(raw_ds))
print("Utasítási példák:", len(inst_ds))
print("Példa prompt:", tok.decode(raw_ds[0]["input_ids"][:40]))

In [ ]:
# 6. A modell NULLÁRÓL: véletlen súlyok, GPT-2-szerű transzformer
import torch
from transformers import GPT2Config, GPT2LMHeadModel

config = GPT2Config(
    vocab_size=VOCAB_SIZE,
    n_positions=N_POSITIONS,
    n_ctx=N_POSITIONS,
    n_embd=N_EMBD,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    bos_token_id=BOS_ID,
    eos_token_id=EOS_ID,
    pad_token_id=PAD_ID,
    resid_pdrop=0.1,
    embd_pdrop=0.1,
    attn_pdrop=0.1,
    use_cache=True,
)

#_fontos_: itt nincs from_pretrained — a modell súlyai véletlenszerűen indulnak.
model = GPT2LMHeadModel(config).cuda()
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
print("Tanítható paraméterek:", round(sum(p.numel() for p in model.parameters()) / 1e6, 1), "M")

In [ ]:
# 7. Tanítás két szakaszban: (1) magyar szöveg, (2) utasítások
from transformers import Trainer, TrainingArguments

class Collator:
    def __call__(self, features):
        n = max(len(f["input_ids"]) for f in features)
        return {
            "input_ids": torch.tensor([f["input_ids"] + [PAD_ID] * (n - len(f["input_ids"])) for f in features]),
            "labels": torch.tensor([f["labels"] + [-100] * (n - len(f["labels"])) for f in features]),
            "attention_mask": torch.tensor([f["attention_mask"] + [0] * (n - len(f["attention_mask"])) for f in features]),
        }

def make_args(stage, lr, epochs):
    return TrainingArguments(
        output_dir=f"/content/nexa-scratch-{stage}",
        num_train_epochs=epochs,
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=lr,
        lr_scheduler_type="cosine",
        warmup_ratio=0.02,
        logging_steps=25,
        save_strategy="no",
        fp16=True,
        optim="adamw_torch",
        gradient_checkpointing=True,
        report_to=[],
    )

print("1. szakasz: magyar szöveges előképzés")
trainer = Trainer(model=model, args=make_args("raw", LR_PRE, PRETRAIN_EPOCHS),
                  train_dataset=raw_ds, data_collator=Collator())
trainer.train()
del trainer
torch.cuda.empty_cache()

print("2. szakasz: utasítás-követő tanítás")
trainer = Trainer(model=model, args=make_args("instruct", LR_INS, INSTRUCT_EPOCHS),
                  train_dataset=inst_ds, data_collator=Collator())
trainer.train()
del trainer
torch.cuda.empty_cache()

In [ ]:
# 8. Gyors ellenőrzés magyarul
model.eval()
prompt = f"System: {SYSTEM}\nUser: Szia, hogy vagy?\nAssistant:"
ids = torch.tensor([tok.encode(prompt).ids], device=model.device)
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=80, do_sample=False,
                         pad_token_id=PAD_ID, eos_token_id=EOS_ID)
print(tok.decode(out[0][ids.shape[1]:].tolist(), skip_special_tokens=True))

In [ ]:
# 9. Mentés: modell + saját tokenizer + chat sablon + generálási beállítások
from transformers import PreTrainedTokenizerFast

OUT = "/content/nexa-scratch"
os.makedirs(OUT, exist_ok=True)

CHAT_TEMPLATE = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}System: {{ message['content'] }}\n"
    "{% elif message['role'] == 'user' %}User: {{ message['content'] }}\n"
    "{% elif message['role'] == 'assistant' %}Assistant: {{ message['content'] }}\n"
    "{% endif %}{% endfor %}"
    "{% if add_generation_prompt %}Assistant:{% endif %}"
)

fast_tok = PreTrainedTokenizerFast(
    tokenizer_object=tok,
    bos_token=BOS_TOK,
    eos_token=EOS_TOK,
    pad_token=PAD_TOK,
    unk_token=UNK_TOK,
    clean_up_tokenization_spaces=False,
)
fast_tok.chat_template = CHAT_TEMPLATE

model.config.use_cache = True
model.save_pretrained(OUT, safe_serialization=True, max_shard_size="2GB")
fast_tok.save_pretrained(OUT)

gc = {
    "max_new_tokens": 256,
    "do_sample": False,
    "bos_token_id": BOS_ID,
    "eos_token_id": EOS_ID,
    "pad_token_id": PAD_ID,
}
with open(os.path.join(OUT, "generation_config.json"), "w", encoding="utf-8") as f:
    json.dump(gc, f, indent=2)

print("Elmentve:", OUT)
for name in sorted(os.listdir(OUT)):
    print(" ", name)

In [ ]:
# 10. ONNX export és 4 bites kvantálás telefonra
import shutil
from pathlib import Path
from optimum.exporters.onnx import main_export
from onnxruntime.quantization import matmul_nbits_quantizer

ONNX_DIR = "/content/nexa-onnx-scratch"
shutil.rmtree(ONNX_DIR, ignore_errors=True)
main_export(
    model_name_or_path=OUT,
    output=ONNX_DIR,
    task="text-generation-with-past",
    device="cpu",
    fp16=False,
    no_post_process=True,
)

for name in ["config.json", "tokenizer.json", "tokenizer_config.json",
             "special_tokens_map.json", "generation_config.json",
             "vocab.json", "merges.txt"]:
    src = os.path.join(OUT, name)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(ONNX_DIR, name))

onnx_dir = Path(ONNX_DIR) / "onnx"
model_path = onnx_dir / "model.onnx"
q_path = onnx_dir / "model_q4.onnx"

quant = matmul_nbits_quantizer.MatMulNBitsQuantizer(
    model=str(model_path), block_size=32, is_symmetric=True, bits=4)
quant.process()
quant.model.save(str(q_path))

model_path.unlink(missing_ok=True)
(model_path.with_name(model_path.name + "_data")).unlink(missing_ok=True)
# A Transformers.js egyes verziói dtype=q4 mellett is a model.onnx nevet keresik.
shutil.copy(q_path, model_path)

size_mb = q_path.stat().st_size / 1048576
print("model_q4.onnx:", round(size_mb, 1), "MB")
assert 5 < size_mb < 900, "Váratlan modellméret"

In [ ]:
# 11. Ellenőrzés: nem üres, véges, van KV-cache, működik a chat sablon
import numpy as np
import onnxruntime as ort

check_prompt = fast_tok.apply_chat_template(
    [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "Szia!"}],
    tokenize=False, add_generation_prompt=True)
ids = np.array([fast_tok.encode(check_prompt).ids], dtype=np.int64)
mask = np.ones_like(ids)

session = ort.InferenceSession(str(q_path), providers=["CPUExecutionProvider"])
inputs = {i.name: i for i in session.get_inputs()}
feed = {"input_ids": ids, "attention_mask": mask}
if "position_ids" in inputs:
    feed["position_ids"] = np.arange(ids.shape[1], dtype=np.int64).reshape(1, -1)
for name, inp in sorted(inputs.items()):
    if name.startswith("past_key_values"):
        shape = [d if isinstance(d, int) else 0 for d in inp.shape]
        if shape and shape[0] == 0:
            shape[0] = 1
        feed[name] = np.zeros(shape, dtype=np.float32)

logits = session.run([o.name for o in session.get_outputs()], feed)[0]
assert np.isfinite(logits).all(), "NaN/Inf érték a kimenetben"
assert logits.max() > logits.min(), "Állandó logits — a modell nem működik"
first = fast_tok.decode([int(np.argmax(logits[0, -1, :]))], skip_special_tokens=True)
print("Következő token:", repr(first))
print("KV-cache kimenetek:", sum(1 for o in session.get_outputs() if o.name.startswith("present")))
print("Chat sablon működik:", check_prompt[:60].replace("\n", " | "))
print("ELLENŐRZÉS OK")

In [ ]:
# 12. Feltöltés a saját Hugging Face repóra
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=OUTPUT_REPO, repo_type="model", private=False, exist_ok=True)

readme = f"""---
library_name: transformers.js
tags: [onnx, transformers.js, hungarian, nexa, scratch, q4]
---
# {OUTPUT_REPO.split("/")[-1]} — nulláról tanítva

**Ez a modell nem finomhangolás.** A súlyokat véletlen értékekről indítottuk,
saját magyar BPE tokenizálóval és saját adatokkal tanítottuk be két szakaszban:
magyar Wikipédia előképzés, majd utasítás-követő tanítás.

- Architektúra: GPT-2-szerű transzformer, {N_LAYER} réteg, {N_EMBD} embedding
- Tokenizáló: saját BPE, {VOCAB_SIZE} token
- Formátum: ONNX, 4 bites `MatMulNBits` (`onnx/model_q4.onnx`)
- Használat: Nexa AI → Beállítások → Saját Nexa modell → `{OUTPUT_REPO}`

A modell telefonon fut, nem küld adatot szerverre. Kis modell lévén a válaszai
rövidek és néha hibásak; ez a teljesen saját súlyok ára.
"""
with open(os.path.join(ONNX_DIR, "README.md"), "w", encoding="utf-8") as f:
    f.write(readme)

api.upload_folder(
    folder_path=ONNX_DIR,
    repo_id=OUTPUT_REPO,
    repo_type="model",
    commit_message="Nexa HU scratch: nullarol betanitott magyar modell (q4 ONNX)",
)
print("Kész: https://huggingface.co/" + OUTPUT_REPO)

## Készen állsz — így használd az appban

1. A Hugging Face repo legyen **nyilvános** (a notebook `private=False`-szel hozza létre).
2. Nexa AI → **Beállítások → Modellek → Saját Nexa modell**: írd be, hogy `sziriusz-blackalt/nexa-hu`.
3. A megjelenő **🧠 Nexa HU** kártyán nyomj a **Letöltés** gombra.
4. A csevegő **Modell** gombjából válaszd a **🧠 Nexa HU**-t.

Ha a modell gyenge, az várható: nulláról, kis paraméterszámmal nem lehet ugyanazt a tudást elérni, mint egy nagy nyílt bázis finomhangolásával. A `nexa_train.ipynb` az erősebb, Qwen3 alapú út.